In [0]:
import time, uuid
from pyspark.sql import functions as F

In [0]:
dbutils.widgets.text("login", "mariiasirska")
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("secret_scope", "lab10-mariiasirska") 
dbutils.widgets.text("k_host", "mariiasirska-postgres-host")
dbutils.widgets.text("k_user", "mariiasirska-postgres-user")
dbutils.widgets.text("k_password", "mariiasirska-postgres-password")
dbutils.widgets.text("db_host_port", "5432")
dbutils.widgets.text("database", "postgres")
dbutils.widgets.dropdown("db_type", "postgresql", ["postgresql", "sqlserver"])
dbutils.widgets.text("foreign_schema", "public")
dbutils.widgets.text("foreign_table", "") 
dbutils.widgets.dropdown("seed_external", "true", ["false", "true"])

In [0]:
login = dbutils.widgets.get("login")
catalog = dbutils.widgets.get("catalog")
bronze = f"{catalog}.{login}_bronze"
silver = f"{catalog}.{login}_silver"
scope = dbutils.widgets.get("secret_scope")
db_type = dbutils.widgets.get("db_type")
database = dbutils.widgets.get("database")
CONN = f"{login}_lab10_{db_type}_conn"
FOREIGN_CAT = f"{login}_lab10_ext"
EXT_SCHEMA = dbutils.widgets.get("foreign_schema")
EXT_TABLE = dbutils.widgets.get("foreign_table") or f"{login}_olist_customers"
FOREIGN_TABLE = f"{FOREIGN_CAT}.{EXT_SCHEMA}.{EXT_TABLE}"
SNAPSHOT = f"{bronze}.lab10_ext_customers_ingested"
print(CONN, "|", FOREIGN_CAT, "|", FOREIGN_TABLE)

In [0]:
print("secret keys in scope:", scope)
print([k.key for k in dbutils.secrets.list(scope) if any(w in k.key.lower() for w in ["postgres", "sql", "host", "user", "pass", "db", "mariia", login.lower()])])
display(spark.sql("SHOW CONNECTIONS"))

In [0]:
k_host, k_user, k_pass = (dbutils.widgets.get(k) for k in ["k_host", "k_user", "k_password"])
assert k_host and k_user and k_pass and database, "Fill k_host, k_user, k_password and database widgets (see discovery cell)"
try:
    spark.sql(f"""
    CREATE CONNECTION IF NOT EXISTS {CONN} TYPE {db_type}
    OPTIONS (
      host secret('{scope}', '{k_host}'),
      port '{dbutils.widgets.get("db_host_port")}',
      user secret('{scope}', '{k_user}'),
      password secret('{scope}', '{k_pass}')
    )""")
    print("connection ready:", CONN)
except Exception as e:
    print("CREATE CONNECTION failed:", str(e)[:300])
    print("-> ask the admin for the CREATE CONNECTION privilege, or reuse an existing connection from SHOW CONNECTIONS")

In [0]:
try:
    spark.sql(f"CREATE FOREIGN CATALOG IF NOT EXISTS {FOREIGN_CAT} USING CONNECTION {CONN} OPTIONS (database '{database}')")
    print("foreign catalog ready:", FOREIGN_CAT)
except Exception as e:
    print("CREATE FOREIGN CATALOG failed:", str(e)[:300])

display(spark.sql(f"SHOW SCHEMAS IN {FOREIGN_CAT}"))

display(spark.sql(f"SHOW TABLES IN {FOREIGN_CAT}.{EXT_SCHEMA}"))

In [0]:
def jdbc_url():
    host = dbutils.secrets.get(scope, k_host)
    ssl = "?sslmode=require" if db_type == "postgresql" else ""
    return f"jdbc:postgresql://{host}:{dbutils.widgets.get('db_host_port')}/{database}{ssl}"

def jdbc_writer(df, mode):
    return (df.write.format("jdbc")
            .option("url", jdbc_url()).option("dbtable", EXT_TABLE)
            .option("user", dbutils.secrets.get(scope, k_user)).option("password", dbutils.secrets.get(scope, k_pass))
            .option("driver", "org.postgresql.Driver").mode(mode))

if dbutils.widgets.get("seed_external") == "true":
    assert db_type == "postgresql", "the JDBC seed helper is written for PostgreSQL"
    seed = spark.sql(f"""
        SELECT customer_id, customer_unique_id, zip_code_prefix, city, state
        FROM {silver}.olist_customers_silver ORDER BY customer_id LIMIT 5000""")
    jdbc_writer(seed, "overwrite").save()
    print("seeded", seed.count(), "rows into", EXT_TABLE)
else:
    print("seed skipped (seed_external = false)")

In [0]:
display(spark.sql(f"SELECT * FROM {FOREIGN_TABLE} LIMIT 10"))
display(spark.sql(f"SELECT state, count(*) AS customers FROM {FOREIGN_TABLE} GROUP BY state ORDER BY customers DESC"))

Federation vs ingestion

In [0]:
t0 = time.time()
spark.sql(f"CREATE OR REPLACE TABLE {SNAPSHOT} AS SELECT * FROM {FOREIGN_TABLE}")
ingest_seconds = round(time.time() - t0, 2)
print(f"one-off ingestion into Delta took {ingest_seconds}s, rows = {spark.table(SNAPSHOT).count()}")

In [0]:
def timed(query, runs=3):
    out = []
    for _ in range(runs):
        t = time.time(); spark.sql(query).collect(); out.append(round(time.time() - t, 2))
    return out

q = "SELECT state, count(*) AS c FROM {t} GROUP BY state ORDER BY c DESC"
fed = timed(q.format(t=FOREIGN_TABLE))
ing = timed(q.format(t=SNAPSHOT))
cmp_df = spark.createDataFrame(
    [("federated (external DB)", *fed), ("ingested (Delta)", *ing)],
    "design string, run1_s double, run2_s double, run3_s double",
)
display(cmp_df)

In [0]:
spark.sql(f"EXPLAIN FORMATTED SELECT state, count(*) FROM {FOREIGN_TABLE} WHERE state = 'SP' GROUP BY state").show(truncate=False)

In [0]:
if dbutils.widgets.get("seed_external") == "true":
    cnt = lambda t: spark.sql(f"SELECT count(*) AS c FROM {t}").first()["c"]
    before = {"federated": cnt(FOREIGN_TABLE), "ingested": cnt(SNAPSHOT)}
    new_id = f"lab10_{uuid.uuid4().hex[:8]}"
    one = spark.createDataFrame([(new_id, "uniq_new", "01000", "Lviv", "SP")],
                                "customer_id string, customer_unique_id string, zip_code_prefix string, city string, state string")
    jdbc_writer(one, "append").save()
    after = {"federated": cnt(FOREIGN_TABLE), "ingested": cnt(SNAPSHOT)}
    print("before:", before)
    print("after a new row in the external DB:", after)
    assert after["federated"] == before["federated"] + 1 and after["ingested"] == before["ingested"]
    spark.sql(f"CREATE OR REPLACE TABLE {SNAPSHOT} AS SELECT * FROM {FOREIGN_TABLE}")
    print("after re-ingestion, ingested =", cnt(SNAPSHOT))
else:
    print("skipped (seed_external = false)")

Joining the external table with Delta tables:

In [0]:
join_sql = f"""
SELECT f.state,
       count(DISTINCT o.order_id) AS orders,
       round(sum(i.price), 2) AS revenue
FROM {FOREIGN_TABLE} f
JOIN {silver}.olist_orders_silver o ON o.customer_id = f.customer_id
JOIN {silver}.olist_order_items_silver i ON i.order_id = o.order_id
WHERE o.order_status NOT IN ('canceled', 'unavailable')
GROUP BY f.state
ORDER BY revenue DESC
LIMIT 10
"""
display(spark.sql(join_sql))


Security evidence

In [0]:
for stmt in [f"DESCRIBE CONNECTION {CONN}", f"SHOW GRANTS ON CONNECTION {CONN}", f"SHOW GRANTS ON CATALOG {FOREIGN_CAT}"]:
    try:
        print("--", stmt)
        display(spark.sql(stmt))
    except Exception as e:
        print("not available:", str(e)[:200])

Lakehouse Federation queries data in place without copying it, so newly inserted rows in PostgreSQL appear immediately, Delta ingestion requires periodic refreshes.

Federated queries are noticeably slower because of network round-trips and external DB execution, even though Spark pushes down WHERE filters and GROUP BY aggregations to PostgreSQL.

Credentials are stored in Azure Key Vault. Access is governed centrally in Unity Catalog via USE CONNECTION and USE CATALOG privileges, the external DB user should be restricted to read-only permissions to prevent accidental writes or heavy load on the operational database.